# Model Comparison: Individual Models vs Ensemble
**Freddie Mac Credit Risk Scoring — SFLLD Sample Dataset**

This notebook compares:
- XGBoost
- LightGBM  
- Random Forest
- Logistic Regression
- Weighted Ensemble (of all above)

Evaluated on both OOS (random split) and OOT (time-based split) test sets.

**Run this AFTER `pipeline.py` has completed at least the `train` stage.**

In [ ]:
import sys
import os
import warnings
warnings.filterwarnings('ignore')

# add project root to path
PROJECT_ROOT = os.path.abspath('')  # assumes notebook is in project root
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from config.settings import CFG
from utils.io_utils import load_dataframe, load_pickle, parquet_exists
from features.engineering import encode_categoricals
from features.splitting import get_xy
from validation.evaluator import (
    compute_all_metrics, generate_comparison_table,
    compute_ks_decile_table, compute_psi
)
from models.scorer import probability_to_score, score_distribution_report

plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})
sns.set_style('whitegrid')
print('Imports OK')

## 1. Load Models and Test Data

In [ ]:
MODEL_DIR    = CFG.paths.model_dir
SPLIT_DIR    = CFG.paths.split_dir
REPORT_DIR   = CFG.paths.report_dir

# load saved artifacts from training stage
encoder      = load_pickle(os.path.join(MODEL_DIR, 'encoder.pkl'))
feature_cols = load_pickle(os.path.join(MODEL_DIR, 'feature_cols.pkl'))

print(f'Encoder loaded: {type(encoder).__name__}')
print(f'Feature columns: {len(feature_cols)} features')
print(f'  First 5: {feature_cols[:5]}')

In [ ]:
# load individual model files
import pickle

MODEL_FILES = {
    'XGBoost':              'xgboost_model.pkl',
    'LightGBM':             'lightgbm_model.pkl',
    'Random Forest':        'rf_model.pkl',
    'Logistic Regression':  'lr_model.pkl',
    'Ensemble':             'ensemble_model.pkl',
}

models = {}
for name, fname in MODEL_FILES.items():
    path = os.path.join(MODEL_DIR, fname)
    if os.path.exists(path):
        with open(path, 'rb') as f:
            models[name] = pickle.load(f)
        print(f'  Loaded: {name}')
    else:
        print(f'  MISSING: {name} ({path})')

print(f'\nTotal models loaded: {len(models)}')

In [ ]:
# categorical columns that need encoding
cat_cols = [c for c in CFG.features.categorical_features if c in feature_cols]

# load test splits
splits_data = {}
for split_name in ['oos', 'oot']:
    path = os.path.join(SPLIT_DIR, f'{split_name}_test.parquet')
    if parquet_exists(path):
        df = load_dataframe(path)
        # apply the same encoder that was used during training
        df_enc, _ = encode_categoricals(df, cat_cols, fit=False, encoder=encoder)
        X = df_enc[feature_cols].fillna(-1)
        y = df_enc['TARGET_12M'].astype(int)
        splits_data[split_name] = (X, y, df_enc)
        print(f'{split_name.upper()} test: {len(df)} loans | default rate: {100*y.mean():.2f}%')
    else:
        print(f'{split_name.upper()} test: NOT FOUND')

if not splits_data:
    raise RuntimeError('No test splits found. Run pipeline.py --stages split first.')

## 2. Compute Metrics for All Models × Splits

In [ ]:
all_metrics = []

for split_name, (X_test, y_test, _) in splits_data.items():
    for model_name, model in models.items():
        try:
            y_score = model.predict_proba(X_test)[:, 1]
            m = compute_all_metrics(y_test.values, y_score, label=f'{model_name}/{split_name}')
            m['model'] = model_name
            m['split'] = split_name
            all_metrics.append(m)
        except Exception as e:
            print(f'ERROR: {model_name} on {split_name}: {e}')

results_df = generate_comparison_table(all_metrics)
print(results_df.to_string(index=False))

In [ ]:
# save the comparison table
os.makedirs(REPORT_DIR, exist_ok=True)
results_df.to_csv(os.path.join(REPORT_DIR, 'model_comparison.csv'), index=False)
print('Comparison table saved.')

## 3. AUC Comparison Bar Chart

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
metric_pairs = [('auc_roc', 'AUC-ROC'), ('ks_stat', 'KS Statistic')]

colors = ['#1E3A5F', '#E63946', '#2A9D8F', '#E9C46A', '#264653']

for ax, (metric, label) in zip(axes, metric_pairs):
    pivot = results_df.pivot(index='model', columns='split', values=metric)
    pivot.plot(kind='bar', ax=ax, color=colors[:len(pivot.columns)], 
               edgecolor='black', width=0.6)
    ax.set_title(f'{label} by Model and Split', fontweight='bold')
    ax.set_xlabel('')
    ax.set_ylabel(label)
    ax.tick_params(axis='x', rotation=30)
    ax.legend(title='Split', loc='lower right')
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter('%.3f'))
    # add value labels on bars
    for container in ax.containers:
        ax.bar_label(container, fmt='%.3f', fontsize=8, padding=2)

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'metric_comparison.png'), 
            dpi=150, bbox_inches='tight')
plt.show()

## 4. ROC Curves

In [ ]:
from sklearn.metrics import roc_curve, precision_recall_curve
from validation.evaluator import compute_auc_roc, compute_gini, compute_auc_pr

# use OOT test if available, else OOS
best_split = 'oot' if 'oot' in splits_data else 'oos'
X_plot, y_plot, _ = splits_data[best_split]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

for i, (name, model) in enumerate(models.items()):
    try:
        y_score = model.predict_proba(X_plot)[:, 1]
        c = colors[i % len(colors)]
        
        auc  = compute_auc_roc(y_plot.values, y_score)
        gini = compute_gini(auc)
        ap   = compute_auc_pr(y_plot.values, y_score)
        
        # ROC
        fpr, tpr, _ = roc_curve(y_plot, y_score)
        ax1.plot(fpr, tpr, lw=2, color=c, 
                 label=f'{name}\n(AUC={auc:.4f}, Gini={gini:.4f})')
        
        # PR
        prec, rec, _ = precision_recall_curve(y_plot, y_score)
        ax2.plot(rec, prec, lw=2, color=c, label=f'{name} (AP={ap:.4f})')
    except Exception as e:
        print(f'Skipping {name}: {e}')

ax1.plot([0,1],[0,1],'k--',lw=1,label='Random')
ax1.set(xlabel='False Positive Rate', ylabel='True Positive Rate',
        title=f'ROC Curves — {best_split.upper()} Test')
ax1.legend(loc='lower right', fontsize=8)
ax1.grid(True, alpha=0.3)

dr = float(y_plot.mean())
ax2.axhline(dr, color='gray', ls='--', lw=1, label=f'Baseline DR={dr:.2%}')
ax2.set(xlabel='Recall', ylabel='Precision',
        title=f'Precision-Recall — {best_split.upper()} Test')
ax2.legend(loc='upper right', fontsize=8)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(REPORT_DIR, 'roc_pr_curves.png'), dpi=150, bbox_inches='tight')
plt.show()

## 5. KS Decile Table — Ensemble on OOT Test

In [ ]:
if 'Ensemble' in models:
    X_plot, y_plot, _ = splits_data[best_split]
    ens_proba = models['Ensemble'].predict_proba(X_plot)[:, 1]
    ks_table = compute_ks_decile_table(y_plot.values, ens_proba)
    
    print(f'KS Decile Table — Ensemble on {best_split.upper()} Test')
    print('=' * 70)
    print(ks_table.to_string(index=False))
    print(f'\nMax KS = {ks_table["ks"].max():.4f} (at decile {ks_table["ks"].idxmax()+1})')
else:
    print('Ensemble model not found.')

## 6. Feature Importance Comparison

In [ ]:
# load feature importance from models that have it
# XGBoost and LightGBM have feature_importances_ attribute
tree_models = {k: v for k, v in models.items() 
               if k in ['XGBoost', 'LightGBM', 'Random Forest']}

if tree_models:
    n_models = len(tree_models)
    fig, axes = plt.subplots(1, n_models, figsize=(6 * n_models, 6))
    if n_models == 1:
        axes = [axes]
    
    for ax, (name, model) in zip(axes, tree_models.items()):
        if hasattr(model, 'feature_importances_'):
            fi = pd.Series(model.feature_importances_, index=feature_cols)
        elif hasattr(model, 'named_steps'):
            # pipeline (LR) - get coefs
            coefs = model.named_steps['lr'].coef_[0]
            fi = pd.Series(np.abs(coefs), index=feature_cols)
        else:
            continue
        
        top15 = fi.nlargest(15)
        top15.sort_values().plot(kind='barh', ax=ax, color='#1E3A5F')
        ax.set_title(f'{name}\nTop 15 Features', fontweight='bold')
        ax.set_xlabel('Importance')
    
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, 'feature_importance.png'), 
                dpi=150, bbox_inches='tight')
    plt.show()

## 7. Credit Score Distribution (300-900 Scale)

In [ ]:
from models.scorer import probability_to_score, score_to_risk_bucket

if 'Ensemble' in models:
    X_plot, y_plot, _ = splits_data[best_split]
    proba  = models['Ensemble'].predict_proba(X_plot)[:, 1]
    scores = probability_to_score(proba)
    
    # show score distribution for defaults vs non-defaults
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
    
    # histogram: scores for default vs non-default
    default_scores     = scores[y_plot.values == 1]
    non_default_scores = scores[y_plot.values == 0]
    
    bins = range(300, 910, 20)
    ax1.hist(non_default_scores, bins=bins, alpha=0.6, color='#2A9D8F', 
             label='Non-Default', density=True)
    ax1.hist(default_scores, bins=bins, alpha=0.6, color='#E63946', 
             label='Default', density=True)
    ax1.set(xlabel='Predicted Credit Score (300-900)',
            ylabel='Density',
            title='Score Distribution by Outcome')
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # default rate by score bucket
    dist_df = score_distribution_report(scores, y_plot.values)
    if 'default_rate_pct' in dist_df.columns:
        ax2.bar(dist_df['score_bucket'], dist_df['default_rate_pct'],
                color='#E63946', edgecolor='black', alpha=0.8)
        ax2.set(xlabel='Score Bucket', ylabel='Default Rate (%)',
                title='Default Rate by Score Bucket')
        ax2.tick_params(axis='x', rotation=30)
        ax2.grid(True, alpha=0.3, axis='y')
        # add value labels
        for bar, val in zip(ax2.patches, dist_df['default_rate_pct']):
            ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.1,
                     f'{val:.1f}%', ha='center', va='bottom', fontsize=9)
    
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, 'score_distribution.png'),
                dpi=150, bbox_inches='tight')
    plt.show()
    
    print('\nScore Distribution Table:')
    print(dist_df.to_string(index=False))

## 8. OOS vs OOT Performance Drop (Stability Check)

If OOT AUC is much lower than OOS AUC, it means the model is not stable over time — it learned patterns specific to the training vintages.

In [ ]:
if 'oos' in splits_data and 'oot' in splits_data:
    print('OOS vs OOT Stability Analysis')
    print('=' * 55)
    print(f'{"Model":<25} {"OOS AUC":>10} {"OOT AUC":>10} {"Drop":>10}')
    print('-' * 55)
    
    oos_metrics = {m['model']: m for m in all_metrics if m['split'] == 'oos'}
    oot_metrics = {m['model']: m for m in all_metrics if m['split'] == 'oot'}
    
    for model_name in models.keys():
        oos_auc = oos_metrics.get(model_name, {}).get('auc_roc', float('nan'))
        oot_auc = oot_metrics.get(model_name, {}).get('auc_roc', float('nan'))
        drop    = oos_auc - oot_auc if not (np.isnan(oos_auc) or np.isnan(oot_auc)) else float('nan')
        flag    = ' *** HIGH DEGRADATION' if (not np.isnan(drop) and drop > 0.05) else ''
        print(f'{model_name:<25} {oos_auc:>10.4f} {oot_auc:>10.4f} {drop:>10.4f}{flag}')
    
    print()
    print('Rule of thumb: drop > 0.05 = model may be overfitting to training vintages')
else:
    print('Need both OOS and OOT test sets for stability analysis.')

## 9. Population Stability Index (PSI)

In [ ]:
if 'oos' in splits_data and 'oot' in splits_data and 'Ensemble' in models:
    # PSI compares the score distribution between OOS (train-era) and OOT (test-era)
    X_oos, y_oos, _ = splits_data['oos']
    X_oot, y_oot, _ = splits_data['oot']
    
    ens = models['Ensemble']
    oos_scores = ens.predict_proba(X_oos)[:, 1]
    oot_scores = ens.predict_proba(X_oot)[:, 1]
    
    psi = compute_psi(oos_scores, oot_scores)
    
    status = 'STABLE' if psi < 0.10 else ('MODERATE SHIFT' if psi < 0.25 else 'SIGNIFICANT SHIFT')
    print(f'PSI (OOS baseline vs OOT current): {psi:.4f} -> {status}')
    
    # score distribution comparison
    fig, ax = plt.subplots(figsize=(10, 5))
    bins = np.linspace(0, 1, 21)
    ax.hist(oos_scores, bins=bins, alpha=0.5, color='#1E3A5F', density=True, label='OOS (train era)')
    ax.hist(oot_scores, bins=bins, alpha=0.5, color='#E63946', density=True, label='OOT (test era)')
    ax.set(xlabel='Predicted Default Probability', ylabel='Density',
           title=f'Score Distribution Shift: OOS vs OOT (PSI={psi:.4f} — {status})')
    ax.legend()
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(os.path.join(REPORT_DIR, 'psi_chart.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Need OOS, OOT, and Ensemble for PSI analysis.')

## 10. Final Summary Table

In [ ]:
print('\n' + '=' * 80)
print('FINAL MODEL COMPARISON SUMMARY')
print('=' * 80)
print(results_df.to_string(index=False))
print()

# which model wins overall?
oot_rows = results_df[results_df['split'] == 'oot'].copy() if 'oot' in results_df['split'].values else results_df.copy()
if len(oot_rows) > 0:
    best_row = oot_rows.sort_values('auc_roc', ascending=False).iloc[0]
    print(f'Best OOT model: {best_row["model"]} | AUC={best_row["auc_roc"]:.4f} | Gini={best_row["gini"]:.4f} | KS={best_row["ks_stat"]:.4f}')